# 05. Heuristic Question Generation & ML Verification

In [ ]:
import os, random, re, joblib
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.sparse import hstack
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
STOP_WORDS = set(stopwords.words('english'))

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models_new")

print("Loading trained verification models...")
tfidf = joblib.load(MODELS_DIR / "tfidf_vectorizer.pkl")
verifier = joblib.load(MODELS_DIR / "lr_balanced.pkl")
dist_ranker = joblib.load(MODELS_DIR / "module_b_rf.pkl")

# Module B models
mod_b_ohe = joblib.load(MODELS_DIR / "module_b_ohe.pkl")
mod_b_rf = joblib.load(MODELS_DIR / "module_b_rf.pkl")
print("Models loaded successfully.")


## 1. Heuristic Generation

In [ ]:
def enumerate_candidate_questions(article, top_k=3):
    sentences = sent_tokenize(str(article))
    if not sentences: sentences = [str(article)]
    try:
        tfidf_local = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1,2), sublinear_tf=True)
        mat = tfidf_local.fit_transform(sentences)
        feature_names = tfidf_local.get_feature_names_out()
    except ValueError:
        mat, feature_names = None, []
    
    scored_sentences = []
    for i, sent in enumerate(sentences):
        sent = sent.replace('\n', ' ').replace('\r', '')
        words = sent.split()
        if len(words) < 4: continue
        score, max_term = 0.0, ""
        if mat is not None and mat.shape[0] > i:
            row = mat.getrow(i)
            if row.nnz > 0:
                max_idx = row.indices[row.data.argmax()]
                score, max_term = row.data.max(), feature_names[max_idx]
        sent_lower = sent.lower()
        if any(cue in sent_lower for cue in ["located", "capital", "city"]): score += 0.05
        is_shortlisted = False
        if any(pat in sent_lower for pat in ["located", "in ", "capital of"]):
            is_shortlisted = True; score += 0.5
        scored_sentences.append((score, is_shortlisted, len(words), i, sent, max_term))
        
    scored_sentences.sort(key=lambda x: (x[1], x[0], x[2]), reverse=True)
    
    candidates, seen = [], set()
    for _, _, _, _, sent, max_term in scored_sentences:
        if len(candidates) >= top_k: break
        answer = ""
        match = re.search(r"is located in ([A-Z][a-z]+(?:\s[A-Z][a-z]+)*)", sent)
        if match: answer = match.group(1)
        elif max_term: answer = max_term
        else:
            tokens = [w for w in re.findall(r"\w+", sent) if w.lower() not in STOP_WORDS]
            answer = max(tokens, key=len) if tokens else sent.split()[-1]
            
        mask_idx = sent.find(answer)
        if mask_idx != -1: masked = sent[:mask_idx] + "_____" + sent[mask_idx+len(answer):]
        else:
            lower_idx = sent.lower().find(answer.lower())
            masked = sent[:lower_idx] + "_____" + sent[lower_idx+len(answer):] if lower_idx != -1 else sent
                
        sent_lower, q_type, template_name = sent.lower(), "What", "cloze"
        if re.search(r"\b(19|20)\d{2}\b", answer) or any(c in sent_lower for c in [" year ", " date "]):
            q_type, template_name = "When", "when"
        elif any(c in sent_lower.split() for c in ["located", "capital", "city"]) or f"in {answer.lower()}" in sent_lower:
            q_type, template_name = "Where", "where"
        elif any(c in sent_lower.split() for c in ["president", "author", "scientist", "he", "she", "man", "woman"]):
            q_type, template_name = "Who", "who"
            
        question = f"According to the passage, what best completes this sentence: {masked}" if template_name == "cloze" else f"{q_type} best completes this: {masked}?"
        key = (question.lower(), answer.lower())
        if key not in seen:
            seen.add(key)
            candidates.append({'question': question, 'correct_answer': answer, 'source_sentence': sent, 'template': template_name})
    return candidates

def generate_heuristic_qa(article):
    cands = enumerate_candidate_questions(article, top_k=1)
    if cands: return cands[0]['question'], cands[0]['correct_answer']
    return None, None


## 2. Option Generation (Module B ML) & Verification

In [ ]:
def char_jaccard(s1, s2):
    set1, set2 = set(str(s1).lower()), set(str(s2).lower())
    if not set1 or not set2: return 0.0
    return len(set1 & set2) / len(set1 | set2)

def get_distractors(article, correct_answer):
    """Module B ML Distractor Generation."""
    tokens = word_tokenize(str(article))
    candidates = list(set(" ".join(tokens[i:i+random.randint(1, 4)]) for i in range(len(tokens)-2)))
    
    ans_lower = correct_answer.lower()
    valid_candidates = [c for c in candidates if c.lower() != ans_lower and c.lower() not in ans_lower and ans_lower not in c.lower()]
    if not valid_candidates: return ["None of the above", "All of the above", "Cannot be determined"]
        
    X_features, art_str = [], str(article).lower()
    for cand in valid_candidates:
        cand_str = cand.lower()
        try:
            vecs = mod_b_ohe.transform([cand_str, ans_lower])
            cos_sim = cosine_similarity(vecs[0], vecs[1])[0,0]
        except: cos_sim = 0.0
        char_match = char_jaccard(cand_str, ans_lower)
        freq = art_str.count(cand_str)
        X_features.append([cos_sim, char_match, freq])
        
    probs = mod_b_rf.predict_proba(np.array(X_features))[:, 1]
    ranked_indices = np.argsort(probs)[::-1]
    
    distractors = [valid_candidates[idx] for idx in ranked_indices[:3]]
    while len(distractors) < 3: distractors.append("None of the above")
    return distractors

def extract_features_for_row(article, question, option, tfidf_model):
    combined = article + " [SEP] " + question + " [SEP] " + option
    tf_vec = tfidf_model.transform([combined])
    art_tokens, q_tokens, opt_tokens = set(article.split()), set(question.split()), set(option.split())
    a_vec, q_vec, o_vec = tfidf_model.transform([article]), tfidf_model.transform([question]), tfidf_model.transform([option])
    sim_aq, sim_ao, sim_qo = cosine_similarity(a_vec, q_vec)[0,0], cosine_similarity(a_vec, o_vec)[0,0], cosine_similarity(q_vec, o_vec)[0,0]
    
    sentences = sent_tokenize(article)
    best_sim, sim_a_best, sim_o_best, sim_q_best = 0.0, 0.0, 0.0, 0.0
    if sentences:
        s_vecs = tfidf_model.transform(sentences)
        sims = cosine_similarity(q_vec, s_vecs)[0]
        best_idx = np.argmax(sims)
        best_sent_vec = s_vecs[best_idx].reshape(1, -1)
        sim_a_best, sim_o_best, sim_q_best = cosine_similarity(a_vec, best_sent_vec)[0,0], cosine_similarity(o_vec, best_sent_vec)[0,0], sims[best_idx]
        
    len_a, len_q, len_o = len(art_tokens), len(q_tokens), len(opt_tokens)
    overlap_qo = len(q_tokens & opt_tokens) / max(1, len_o)
    overlap_ao = len(art_tokens & opt_tokens) / max(1, len_o)
    exact_match = 1 if option in article else 0
    freq = article.count(option)
    num_vec = np.array([[sim_aq, sim_ao, sim_qo, sim_a_best, sim_o_best, sim_q_best, len_a, len_q, len_o, overlap_qo, overlap_ao, exact_match, freq]])
    return hstack([tf_vec, num_vec])

def end_to_end_demo(article):
    print("--- Original Article ---")
    print(article[:300] + "...\n")
    question, correct_answer = generate_heuristic_qa(article)
    if not question: return "Failed to generate question."
    print(f"Generated Question: {question}")
    
    distractors = get_distractors(article, correct_answer)
    options = [correct_answer] + distractors
    random.shuffle(options)
    
    print("\n--- Options (Module B Distractors) ---")
    for i, opt in enumerate(options):
        print(f"{chr(65+i)}: {opt}")
        
    best_score, best_idx = -np.inf, 0
    for i, opt in enumerate(options):
        features = extract_features_for_row(article, question, opt, tfidf)
        score = verifier.predict_proba(features)[0, 1]
        if score > best_score:
            best_score = score
            best_idx = i
            
    print(f"\n=> ML Verifier picked Option {chr(65+best_idx)} (Score: {best_score:.4f})")
    correct_idx = options.index(correct_answer)
    if best_idx == correct_idx: print("✅ Correct! Verifier successfully identified the heuristic answer.")
    else: print(f"❌ Incorrect. True answer was {chr(65+correct_idx)}.")


In [ ]:
# Demo
val_df = pd.read_csv(PROCESSED_DIR / "val_verification.csv")
sample_article = val_df['article'].iloc[15]
end_to_end_demo(sample_article)
